# These Functions convert the Dataset into a YOLO Format

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
#automatically reload any imported modules when you re-run a cell

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path (so imports like src.data work)
project_root = Path().cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import xywhr_to_xyxyxyxy
from src import logger
from src import DATA_BASE_DIR as data_base_dir

In [ ]:
import json
import os
import numpy as np
import time
import os
import shutil
import zipfile
import yaml
import re
import cv2
import shutil

In [ ]:
#yolo_coco_id_match = {"person":0, "clothing":1, "bag":2}
#yolo_coco_id_match_action = {"seated":0, "clothing":1, "bag":2, "seated_ground":3, "standing":4, "lying":5}
#yolo_coco_id_match_action = {"seated":0, "seated_ground":1, "standing":2, "lying":3, "clothing":4, "bag":5}
#yolo_coco_id_match_action = {"seated":0, "standing":1, "lying":2, "seated_ground":3, "clothing":4, "bag":5}

def coco_to_yolo(data_base_dir, record_id, restricted_classes=[], action_labels=False):
    """
    Convert COCO-format JSON annotations to YOLO .txt files.

    """
    if action_labels==True:
        yolo_coco_id_match = {"seated":0, "standing":1, "lying":2, "seated_ground":3, "clothing":4, "bag":5}
    else:
        yolo_coco_id_match = {"person":0, "clothing":1, "bag":2}
    print(f'Using: {yolo_coco_id_match}')
    
    output_dir = Path(data_base_dir) / "labels" / record_id
    output_dir.mkdir(parents=True, exist_ok=True)

    annotation_path = Path(data_base_dir) / "annotations" / f"{record_id}_annotations.json"
    with open(annotation_path, "r") as f:
        all_annotations = json.load(f)

    images = {str(img["id"]): img for img in all_annotations["images"]}

    category_id_name_match = {cat["id"]:cat["name"] for cat in all_annotations["categories"]}
    print(category_id_name_match)
    
    anns_by_image = {}
    for ann in all_annotations["annotations"]:
        img_id = str(ann["image_id"]) #this image_id matches the image_id in the dictionary "images"
        anns_by_image.setdefault(img_id, []).append(ann) #setdefault() ensures a list exists for that image_id, then adds the annotation to that image’s list
 
    # Process each image
    for image_id in list(images.keys()):
        
        start = time.time()
        image_data = images.get(image_id)
        if not image_data:
            continue

        image_width = image_data["width"]
        image_height = image_data["height"]
        file_stem = image_data["file_name"].replace(".png", "")

        anns = anns_by_image.get(image_id, [])
        if not anns:
            continue

        yolo_annotations = []
        for ann in anns:
            attributes = ann.get("attributes", {})
            occluded=attributes["occluded"]
            cls_name = category_id_name_match[ann["category_id"]]
            
            if action_labels:
                if cls_name != 'person':
                    continue
                #using the action attribute as class instead of the class "person"      
                action=attributes.get("action", None)
                #if action=="seated_ground":
                #    action="lying"
                #    print("seated_ground -> lying")
                cls_name = action
                    
            #match cls_name to yolo category id
            yolo_cat_id = yolo_coco_id_match[cls_name]
            #print(cls_name, yolo_cat_id)
            
            if not occluded and yolo_cat_id not in restricted_classes:
                bbox=ann["bbox"]
                rotation=attributes["rotation"]
                xyxyxyxy=xywhr_to_xyxyxyxy(bbox, rotation)            
                xyxyxyxy_n = [xyxyxyxy[0]/image_width, xyxyxyxy[1]/image_height, 
                              xyxyxyxy[2]/image_width, xyxyxyxy[3]/image_height, 
                              xyxyxyxy[4]/image_width, xyxyxyxy[5]/image_height, 
                              xyxyxyxy[6]/image_width, xyxyxyxy[7]/image_height]
                
                annotation_line = f"{yolo_cat_id} " + " ".join(f"{coord:.6f}" for coord in xyxyxyxy_n)
                yolo_annotations.append(annotation_line)

        with open(output_dir / f"{file_stem}.txt", "w") as txt_file:
            txt_file.write("\n".join(yolo_annotations))
        
        end = time.time()
        print(f"Processing image {image_id} took {end-start:.4f} sec")


In [ ]:
from src.data import list_record_ids
record_ids = list_record_ids()

In [ ]:
record_ids

In [ ]:
for record_id in record_ids:
    print(f"Starting with record id: {record_id}")
    coco_to_yolo(data_base_dir, record_id, action_labels=False, restricted_classes = [1,2]) 
    ##for just person: restricted_classes = [1, 2]
    ##using action labels, but not bag or clothing: restricted_classes = [4, 5]

# Generate train.txt & val.txt Files

In [ ]:
def generate_data_txt_for_sequences(base_dirs_with_sequences, output_file="val.txt", max_entries=None):
    """
    Generate txt listing images (with labels) only for specified sequences.
    
    Parameters
    ----------
    base_dirs_with_sequences : list of (str, list of str)
        Example: [(base_dir1, ["seq1","seq2"]), (base_dir2, ["seqA","seqB"]), ...]
    output_file : str
        Output txt filename
    max_entries : int, optional
        Limit number of entries, if one wants to run on a subset

    Returns
    -------
    nr_entries: int
        Number of entries
    output_file: str
        Path to output_file
    
    Raises
    ------
    FileNotFoundError
    """
    valid_entries = []

    for base_dir, sequences in base_dirs_with_sequences:
        labels_root = os.path.join(base_dir, "labels")
        images_root = os.path.join(base_dir, "images")
        
        for seq in sequences:
            # Check, if the label and image directories exist.
            labels_seq_dir = os.path.join(labels_root, seq)
            images_seq_dir = os.path.join(images_root, seq)
    
            if not os.path.isdir(labels_seq_dir) or not os.path.isdir(images_seq_dir):
                raise FileNotFoundError(f"Missing image or label directory for {seq} in {base_dir}")
            logger.info(f"Found {labels_seq_dir} and {images_seq_dir}")


            for img_file in sorted(os.listdir(images_seq_dir)):
                if not img_file.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                    
                image_path = os.path.join(images_seq_dir, img_file)
                label_path = os.path.join(labels_seq_dir, os.path.splitext(img_file)[0] + ".txt")
            
                if not os.path.exists(label_path):
                    # Create empty txt for background images
                    with open(label_path, "x") as f:
                        pass

                valid_entries.append(os.path.abspath(image_path))
                if max_entries is not None and len(valid_entries)>=max_entries:
                    break

    nr_entries = len(valid_entries)
    if nr_entries == 0:
        raise FileNotFoundError(f"No entries for {output_file}.")    
    
    with open(output_file, "w") as f:
        f.write('\n'.join(valid_entries))
    
    print(f"{output_file} created with {nr_entries} entries at {output_file}")
    return nr_entries, output_file

In [ ]:
datasets_train = [
    (data_base_dir, [f"rec{str(i)}" for i in range(1, 26)])] # TRAIN

datasets_val = [
    (data_base_dir, ["rec26", "rec27", "rec28", "rec29", "rec30"])] # VAL

generate_data_txt_for_sequences(datasets_train, output_file=f"{data_base_dir}/train.txt")
generate_data_txt_for_sequences(datasets_val, output_file=f"{data_base_dir}/val.txt")

In [ ]:
HABBOF_BASE_DIR = "/home/stella/computer_vision/fisheye_data"
Hval = [(HABBOF_BASE_DIR, ["Lab1", "Lab2", "Meeting1", "Meeting2"])]
generate_data_txt_for_sequences(Hval, output_file=f"{HABBOF_BASE_DIR}/H_val.txt")


# Write yaml-file

In [ ]:
import yaml

yaml_text = dict(
    path = str(data_base_dir),
    train = 'train.txt',
    val = 'val.txt',
    names = {0: 'person',
             1: 'clothing',
             2: 'bag'}
)

with open(f'{data_base_dir}/PMOF.yaml', 'w') as outfile:
    yaml.dump(yaml_text, outfile, default_flow_style=False, sort_keys=False)